# Generate raster data

This notebook builds plate-model-based rasters (seafloor age, sediment and carbonate thickness, crustal thickness, CO₂, erosion/deposition) under the extracted-data tree, using the selected run config.

I encourage running `python3 run_notebooks.py --cache-remote --config [CONFIG]` via CLI in the project root before running any of these notebooks. This will ensure that all necessary remote files are downloaded in advance, and will populate `config/.run_config.yml` with the desired config file (meaning you don't have to unecessarily fiddle with individual notebooks).

>**WARNING:** Many of the raster construction protocols herein were designed specifically for the modified version of the Clennett 2020 reconstruction used in the Alfonso 2024 et al. paper, from which this codebase is derived. They have not been tested for new plate reconstructions (e.g. Zahirovic2022). Use with caution.

## Notebook setup

Paths, timespan, and feature flags come from the config file and are loaded into `PathConfigManager`; optional feature blocks run only when enabled in config `feature_sets`.

In [1]:
config_file = "config/.run_config.yml"

In [2]:
from lib.paths import PathConfigManager
pcm = PathConfigManager(config_file, notebook="00a")

# =====================
# Filestructure
# =====================

plate_model_dir = pcm.PLATE_MODEL_DIR
raster_output_dir = pcm.RASTER_DATA_DIR

pcm.create_directories()

# =====================
# Notebook scope
# =====================

# Gates for determining scope of notebook run (i.e. which features to generate)
use_features = pcm.use_features

# =====================
# Plate model
# =====================

# Plate model
plate_model_name = pcm.config["plate_model"]["plate_model_name"]
use_provided_plate_model = pcm.config["plate_model"]["use_provided_plate_model"]

# Timespan for analysis
min_time = pcm.config["timespan"]["min"]
max_time = pcm.config["timespan"]["max"]
times = range(min_time, max_time + 1)

# =====================
# Extraction parameters
# =====================

# Whether to overwrite previous rasters (default = False)
overwrite = pcm.config["overwrite_output"]

# Number of processes to use
n_jobs = pcm.config["n_jobs"]

# Whether to cleanup working directories (default = False)
cleanup = pcm.config["cleanup"]

In [3]:
import multiprocessing
import shutil
import sys
import warnings
from pathlib import Path

import geopandas as gpd
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    import gplately
import joblib
import numpy as np
import pandas as pd
import pygplates
import rioxarray
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from gplately import PlotTopologies

from lib.check_files import check_plate_model
from lib.extract_data import convert_age_to_depth
from lib.extract_data.create_lip_conjugates import create_lip_conjugates
from lib.extract_data.crustal_co2 import calculate_crustal_co2
from lib.extract_data.crustal_thickness import calculate_crustal_thickness
from lib.extract_data.lip_reconstruction import (
    calculate_depth_contours,
    mp_wrapper_for_shapefile_to_raster,
    rotation_LIP_shapefile_and_buffer,
)
from lib.extract_data.paleobathymetry import (
    # add_Pacific_synthetic_seamounts,
    calculate_paleobathymetry,
)
from lib.extract_data.paleotopography import paleotopography_job
from lib.extract_data.paleotopography.create_present_day_features import create_present_day_features
from lib.extract_data.paleotopography.tween_paleoshorelines_inplace import tween_paleoshorelines_inplace
from lib.plate_models import get_plate_reconstruction, get_plot_topologies

# Sediment thickness modules
sedthick_path = Path("submodules") / "predicting-sediment-thickness"
if str(sedthick_path) not in sys.path:
    sys.path.insert(0, str(sedthick_path))
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from ocean_basin_proximity import (
        generate_and_write_proximity_data_parallel,
        generate_input_points_grid as generate_input_points_grid_seds,
    )
    from predict_sediment_thickness import (
        predict_sedimentation,
        write_grd_file,
    )

# Carbonate thickness modules
carbonate_repo = Path("submodules") / "CarbonateSedimentThickness"
if str(carbonate_repo) not in sys.path:
    sys.path.insert(0, str(carbonate_repo))
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from carbonate_sediment_thickness import (
        calc_sedimentation,
        generate_input_points_grid as generate_input_points_grid_carb,
        write_data,
    )

*^^ Runtime: 26.1 sec*

In [4]:
if use_provided_plate_model:
    check_plate_model(plate_model_dir, verbose=True)
    plate_model_name = None

plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

if use_provided_plate_model:
    coastlines_filenames = [
        str(
            plate_model_dir
            / "StaticGeometries"
            / "AgeGridInput"
            / "CombinedTerranes.gpml"
        )
    ]
    gplot = PlotTopologies(
        plate_model,
        coastlines=coastlines_filenames,
        continents=coastlines_filenames,
    )

else:
    gplot = get_plot_topologies(
        model_name=plate_model_name,
        model_dir=plate_model_dir,
        plate_reconstruction=plate_model,
    )

2026-04-07,17:29:54 - pmm - WARNING - Unable to fetch https://repo.gplates.org/webdav/pmm/config/models_v2.json.
2026-04-07,17:29:54 - pmm - WARNING - Unable to fetch https://www.earthbyte.org/webdav/pmm/config/models_v2_eb.json.
2026-04-07,17:29:54 - pmm - WARNING - Unable to fetch https://portal.gplates.org/static/pmm/config/models_v2_gp.json.


## Seafloor age

In [ ]:
if use_features('subduction'):
    
    seafloor_age_output_dir = raster_output_dir / "SeafloorAge"
    spreading_rate_output_dir = raster_output_dir / "SpreadingRate"

    run_seafloor_age = overwrite
    if not run_seafloor_age:
        for time in times:
            fp = seafloor_age_output_dir / f"seafloor_age_{time:0.0f}Ma.nc"
            q = spreading_rate_output_dir / f"spreading_rate_{time:0.0f}Ma.nc"
            if not (fp.exists() and q.exists()):
                run_seafloor_age = True
                break

    if run_seafloor_age:
        # gplately uses multiprocessing.cpu_count() - 1 internally for MOR seeding.
        # Override it so worker count follows n_jobs and does not oversubscribe.
        old_cpu_count = multiprocessing.cpu_count
        multiprocessing.cpu_count = lambda: n_jobs + 1

        try:
            seafloor_grid = gplately.SeafloorGrid(
                gplot.plate_reconstruction,
                gplot,
                min_time=min_time,
                max_time=max_time,
                ridge_time_step=1,
                save_directory=raster_output_dir / "seafloor_age_output",
                resume_from_checkpoints=True
            )
            seafloor_grid.reconstruct_by_topologies()
        finally:
            multiprocessing.cpu_count = old_cpu_count

        for i in ["SEAFLOOR_AGE", "SPREADING_RATE"]:
            seafloor_grid.lat_lon_z_to_netCDF(
                i,
                nprocs=n_jobs,
            )

        for which in "SEAFLOOR_AGE", "SPREADING_RATE":
            d_old = raster_output_dir / "seafloor_age_output" / which
            d_new = raster_output_dir / "".join([i.capitalize() for i in which.split("_")])
            d_new.mkdir(parents=True, exist_ok=True)
            for t in times:
                src = f"{which}_grid_{t:0.2f}Ma.nc"
                dst = f"{which.lower()}_{t:0.0f}Ma.nc"
                (d_old / src).rename(d_new / dst)
            d_old.rmdir()
        shutil.rmtree(raster_output_dir / "seafloor_age_output")

## Sediment thickness

In [ ]:
if use_features('subduction'):
    sedthick_output_dir = raster_output_dir / "SedimentThickness"
    sedthick_output_dir.mkdir(parents=True, exist_ok=True)

    sedthick_workdir = raster_output_dir / "sedthick_outputs"
    sedthick_workdir.mkdir(parents=True, exist_ok=True)

    # Values taken from predicting-sediment-thickness scripts
    initial_grid_spacing = 1.0
    final_grid_spacing = 0.1
    max_mean_distance = 3000
    max_age = 191.87276
    mean_age = 61.18406823
    mean_distance = 1835.28118479
    variance_age = 1934.6999014
    variance_distance = 1207521.8995806
    age_distance_polynomial_coefficients = [
        5.441401190368497, 0.46893096, -0.07320928, -0.24077496, -0.10840657,
        0.00381672, 0.06831728, 0.01179914, 0.01158149, -0.39880562,
    ]

    if overwrite:
        missing_sed_times = list(times)
        existing_sed_times = []
    else:
        missing_sed_times = []
        existing_sed_times = []
        for time in times:
            sed_file = sedthick_output_dir / f"sediment_thickness_{time:0.0f}Ma.nc"
            if sed_file.exists():
                existing_sed_times.append(time)
            else:
                missing_sed_times.append(time)

    # Only care about missing mean-distance grids for times still needing sediment outputs.
    missing_mean_times = []
    for time in missing_sed_times:
        mean_file = sedthick_workdir / f"mean_distance_{final_grid_spacing:0.1f}d_{time:0.1f}.nc"
        if not mean_file.exists():
            missing_mean_times.append(time)

    print(
        "Sediment resume check: "
        f"existing={len(existing_sed_times)}, "
        f"missing={len(missing_sed_times)}, "
        f"missing_mean_distance_for_missing_sed={len(missing_mean_times)}"
    )

    run_sedthick = bool(missing_sed_times)

    if run_sedthick:
        input_points_initial = generate_input_points_grid_seds(initial_grid_spacing)[0]
        input_points_final = generate_input_points_grid_seds(final_grid_spacing)[0]

        if use_provided_plate_model:
            sedthick_cobs = [
                plate_model_dir / "North_America_COBs.gpml",
                plate_model_dir / "StaticGeometries" / "AgeGridInput" / "Global_EarthByte_GeeK07_COB_Terranes.gpml",
            ]
            coastline_files = coastlines_filenames
        else:  # Use PlateModel from the PlateReconstruction to access files directly
            model = plate_model.plate_model
            sedthick_cobs = model.get_COBs(True) or model.get_continental_polygons(True)
            coastline_files = model.get_coastlines()

        if missing_mean_times:
            missing_seafloor_age_times = []
            seafloor_age_filenames = []
            for time in missing_mean_times:
                age_file = raster_output_dir / "SeafloorAge" / f"seafloor_age_{time:0.0f}Ma.nc"
                if not age_file.exists():
                    missing_seafloor_age_times.append(time)
                    continue
                seafloor_age_filenames.append((age_file, time))

            if missing_seafloor_age_times:
                raise FileNotFoundError(
                    "Missing SeafloorAge files required to build proximity grids for sediment resume. "
                    f"Times: {missing_seafloor_age_times[:20]}"
                )

            generate_and_write_proximity_data_parallel(
                input_points=input_points_initial,
                rotation_filenames=model.get_rotation_model(),
                proximity_filenames=sedthick_cobs,
                proximity_features_are_topological=False,
                proximity_feature_types=None,
                topological_reconstruction_filenames=model.get_topologies(),
                age_grid_filenames_and_paleo_times=seafloor_age_filenames,
                time_increment=1,
                output_distance_with_time=True,
                output_mean_distance=True,
                output_standard_deviation_distance=True,
                output_directory=sedthick_workdir,
                max_topological_reconstruction_time=None,
                continent_obstacle_filenames=coastline_files,
                anchor_plate_id=0,
                proximity_distance_threshold_radians=None,
                clamp_mean_proximity_distance_radians=max_mean_distance / gplately.EARTH_RADIUS,
                output_grd_files=(initial_grid_spacing, final_grid_spacing),
                num_cpus=n_jobs,
            )

        sedthick_output_template = sedthick_output_dir / r"sediment_thickness_{:0.0f}Ma.nc"
        age_grid_filename_template = raster_output_dir / "SeafloorAge" / r"seafloor_age_{:0.0f}Ma.nc"
        distance_grid_filename_template = sedthick_workdir / (
            f"mean_distance_{final_grid_spacing:0.1f}d"
            + r"_{:0.1f}.nc"
        )

        def func(
            time,
            output_filename,
            input_points,
            age_grid_filename,
            distance_grid_filename,
            mean_age,
            mean_distance,
            variance_age,
            variance_distance,
            age_distance_polynomial_coefficients,
            max_age,
            max_distance,
            grid_spacing,
        ):
            if not distance_grid_filename.is_file():
                raise FileNotFoundError(
                    "Missing required mean-distance grid for sediment thickness: "
                    f"{distance_grid_filename} (time={time} Ma)"
                )

            result = predict_sedimentation(
                input_points=input_points,
                age_grid_filename=age_grid_filename,
                distance_grid_filename=distance_grid_filename,
                mean_age=mean_age,
                mean_distance=mean_distance,
                variance_age=variance_age,
                variance_distance=variance_distance,
                age_distance_polynomial_coefficients=age_distance_polynomial_coefficients,
                max_age=max_age,
                max_distance=max_distance,
            )
            write_grd_file(
                output_filename,
                output_data=result,
                grid_spacing=grid_spacing,
                num_grid_longitudes=None,  # not used
                num_grid_latitudes=None,  # not used
            )

        with joblib.Parallel(n_jobs) as parallel:
            parallel(
                joblib.delayed(func)(
                    time=time,
                    output_filename=sedthick_output_template.format(time),
                    input_points=input_points_final,
                    age_grid_filename=age_grid_filename_template.format(time),
                    distance_grid_filename=distance_grid_filename_template.format(time),
                    mean_age=mean_age,
                    mean_distance=mean_distance,
                    variance_age=variance_age,
                    variance_distance=variance_distance,
                    age_distance_polynomial_coefficients=age_distance_polynomial_coefficients,
                    max_age=max_age,
                    max_distance=max_mean_distance,
                    grid_spacing=final_grid_spacing,
                )
                for time in missing_sed_times
            )
    else:
        print("All sediment thickness grids already exist; skipping sediment thickness generation.")

## Carbonate thickness

In [ ]:
if use_features('subduction'):
    carbonate_output_dir = raster_output_dir / "CarbonateThickness"
    carbonate_output_dir.mkdir(parents=True, exist_ok=True)

    carbonate_workdir = raster_output_dir / "carbonate_outputs"
    carbonate_workdir.mkdir(parents=True, exist_ok=True)

### Paleobathymetry

In [ ]:
if use_features('subduction'):
    run_paleobath = overwrite
    if not run_paleobath:
        for time in times:
            fp = carbonate_workdir / f"paleobathymetry_{time:0.0f}Ma.nc"
            if not fp.exists():
                run_paleobath = True
                break

    if run_paleobath:
        # Basement depth
        def func(time, input_dir, output_dir):
            agegrid_filename = Path(input_dir) / f"seafloor_age_{time:0.0f}Ma.nc"
            output_filename = Path(output_dir) / f"basement_depth_{time:0.0f}Ma.nc"

            agegrid = gplately.Raster(str(agegrid_filename))
            basement_depth = agegrid.copy()
            basement_depth.data = convert_age_to_depth(agegrid.data)
            basement_depth.save_to_netcdf4(str(output_filename))


        with joblib.Parallel(n_jobs) as parallel:
            parallel(
                joblib.delayed(func)(
                    time=time,
                    input_dir=raster_output_dir / "SeafloorAge",
                    output_dir=carbonate_workdir,
                )
                for time in times
            )

        # Large igneous provinces and seamounts
        paleobath_data_dir = Path("lib") / "extract_data" / "paleobath_data"
        gdf_seamounts = gpd.read_file(
            paleobath_data_dir
            / "Seamount_shapefile"
            / "Johansson_etal_2018_VolcanicProvinces_v2_NW-edit.shp"
        )
        gdf_LIPs = gpd.read_file(
            paleobath_data_dir / "LIP_shapefile" / "LIPs_merged.shp"
        )
        gdf_features = gpd.pd.concat([gdf_LIPs, gdf_seamounts])

        basement_depth = rioxarray.open_rasterio(
            str(carbonate_workdir / "basement_depth_0Ma.nc"),
            masked=True,
        )
        basement_depth = basement_depth.sel(band=1)
        basement_depth = basement_depth.drop(['band']) #, 'spatial_ref']) # spatial ref appears nonexistent
        basement_depth.rio.write_crs("epsg:4326", inplace=True)  # set crs

        (carbonate_workdir / "GDH1").mkdir(parents=True, exist_ok=True)

        # LIPs
        calculate_depth_contours(
            basement_depth,
            gdf_features,
            contour_interval=50,
            LIP_depth_rounding=50,
            LIP_output_dir=carbonate_workdir,
            clip_contour_to_outline="yes",
            LIP_model_name="GDH1",
            path_contoured_LIPs=carbonate_workdir,
            feature_type="LIPs_seamounts",
            gdf_LIP_large=None,
        )

        # Conjugate LIPs
        create_lip_conjugates(
            LIP_contour_poly=str(
                carbonate_workdir / "contoured_LIPs_seamounts_0Ma.shp"
            ),
            rotation_model=plate_model.rotation_model.filenames,
            topology_features=plate_model.topology_features.filenames,
            LIP_conjugate_outdir=carbonate_workdir,
        )

        path_rotated_polygons = carbonate_workdir / "GDH1" / "rotated_polygons"
        path_rotated_polygons.mkdir(parents=True, exist_ok=True)
        # Working on 0 Ma ONLY.
        # This is because we need to create swell_stats.txt (which is only created at 0 Ma)
        rotation_LIP_shapefile_and_buffer(
            time=0.0,
            path_0_shp=str(
                carbonate_workdir / "contoured_LIPs_seamounts_0Ma.shp"
            ),
            rotation_filenames=plate_model.rotation_model.filenames,
            path_rotated_polygons=str(path_rotated_polygons),
            feature_type="LIPs_seamounts",
            include_conjugate_LIPs="yes",
            LIP_output_dir=carbonate_workdir,
            LIP_model_name="GDH1",
            cooling_model="GDH1",
            large_LIP_path=None,
            buffer_radius_deg=1.0,
            RHCW_age_depth_interp=None,
        )

        with joblib.Parallel(n_jobs) as parallel:
            parallel(
                joblib.delayed(rotation_LIP_shapefile_and_buffer)(
                    time=time,
                    path_0_shp=str(
                        carbonate_workdir / "contoured_LIPs_seamounts_0Ma.shp"
                    ),
                    rotation_filenames=plate_model.rotation_model.filenames,
                    path_rotated_polygons=str(path_rotated_polygons),
                    feature_type="LIPs_seamounts",
                    include_conjugate_LIPs="yes",
                    LIP_output_dir=carbonate_workdir,
                    LIP_model_name="GDH1",
                    cooling_model="GDH1",
                    large_LIP_path=None,
                    buffer_radius_deg=1.0,
                    RHCW_age_depth_interp=None,
                )
                for time in times
            )
            parallel(
                joblib.delayed(mp_wrapper_for_shapefile_to_raster)(
                    time=time,
                    path_rotated_polygons=str(path_rotated_polygons),
                    feature_type="LIPs_seamounts",
                    cooling_model="GDH1",
                    grid_spacing=0.1,
                    lon_min=-180,
                    lon_max=180,
                    lat_min=-90,
                    lat_max=90,
                    path_output_grids=carbonate_workdir,
                )
                for time in times
            )

            # Create combined paleobathymetry
            parallel(
                joblib.delayed(calculate_paleobathymetry)(
                    sedthick_filename=str(
                        raster_output_dir
                        / "SedimentThickness"
                        / f"sediment_thickness_{time:0.0f}Ma.nc"
                    ),
                    basement_depth_filename=str(
                        carbonate_workdir
                        / f"basement_depth_{time:0.0f}Ma.nc"
                    ),
                    lip_height_filename=str(
                        carbonate_workdir
                        / f"reconstructed_LIPs_seamounts_{time:0.0f}Ma.nc"
                    ),
                    output_filename=str(
                        carbonate_workdir
                        / f"paleobathymetry_{time:0.0f}Ma.nc"
                    ),
                )
                for time in times
            )

### Carbonate thickness

In [ ]:
if use_features('subduction'):
    run_carbonate = overwrite
    if not run_carbonate:
        for time in times:
            fp = carbonate_output_dir / f"carbonate_thickness_{time:0.0f}Ma.nc"
            if not fp.exists():
                run_carbonate = True
                break

    if run_carbonate:
        ccd_curve_filename = str(
            carbonate_repo / "input_data" / "Boss_Wilkinson_1991_global_CCD.txt"
        )
        max_carbonate_decomp_sed_rate_cm_per_ky_curve_filename = str(
            carbonate_repo / "input_data" / "sed_rate_v6.txt"
        )

        # Create symlinks for seafloor age and bathymetry files
        # (necessary for carbonate thickness function)
        for time in times:
            # Seafloor age
            link_path = raster_output_dir / "SeafloorAge" / f"seafloor_age_{time:0.0f}.nc"
            target_path = raster_output_dir / "SeafloorAge" / f"seafloor_age_{time:0.0f}Ma.nc"
            if link_path.is_file():
                link_path.unlink()
            link_path.symlink_to(target_path.relative_to(link_path.parent))

            # Bathymetry
            link_path = carbonate_workdir / f"paleobathymetry_{time:0.0f}.nc"
            target_path =carbonate_workdir / f"paleobathymetry_{time:0.0f}Ma.nc"
            if link_path.is_file():
                link_path.unlink()
            link_path.symlink_to(target_path.relative_to(link_path.parent))

        grid_spacing = 0.5
        latitude_range = (-90, 90)
        longitude_range = (-180, 180)
        input_points = generate_input_points_grid_carb(
            grid_spacing_degrees=grid_spacing,
            latitude_range=latitude_range,
            longitude_range=longitude_range,
        )
        age_grid_filename_components = (
            str(raster_output_dir / "SeafloorAge" / "seafloor_age_"),  # prefix
            0,  # decimal places in time component
            "nc",  # file extension
        )
        bathymetry_filename_components = (
            str(carbonate_workdir / "paleobathymetry_"),  # prefix
            0,  # decimal places in time component
            "nc",  # file extension
        )


        def func(output_dir, grid_spacing, latitude_range, longitude_range, **kwargs):
            kwargs["rotation_model"] = pygplates.RotationModel(kwargs["rotation_model"])
            sediment_thickness_data = calc_sedimentation(**kwargs)
            (carbonate_decompacted_sediment_thickness_data,
            carbonate_compacted_sediment_thickness_data,
            carbonate_deposition_mask_data) = sediment_thickness_data
            time = kwargs.get("time")
            output_prefix = str(
                Path(output_dir) / f"carbonate_thickness_{time:0.0f}Ma"
            )
            write_data(
                data=carbonate_decompacted_sediment_thickness_data,
                output_filename_prefix=output_prefix,
                grid_spacing=grid_spacing,
                latitude_range=latitude_range,
                longitude_range=longitude_range,
            )


        with joblib.Parallel(n_jobs) as parallel:
            parallel(
                joblib.delayed(func)(
                    output_dir=carbonate_output_dir,
                    grid_spacing=grid_spacing,
                    latitude_range=latitude_range,
                    longitude_range=longitude_range,
                    input_points=input_points,
                    age_grid_filename_components=age_grid_filename_components,
                    bathymetry_filename_components=bathymetry_filename_components,
                    bathymetry_filename_oldest_time=max(times),
                    topology_filenames=plate_model.topology_features.filenames,
                    rotation_model=plate_model.rotation_model.filenames,
                    ccd_curve_filename=ccd_curve_filename,
                    max_carbonate_decomp_sed_rate_cm_per_ky_curve_filename=max_carbonate_decomp_sed_rate_cm_per_ky_curve_filename,
                    carbonate_anchor_plate_id=0,
                    time=time,
                )
                for time in times
            )

        # Clean up .xy files and symlinks for seafloor age and bathymetry files
        for time in times:
            # .xy carbonate thickness files
            xy_path = carbonate_output_dir / f"carbonate_thickness_{time:0.0f}Ma.xy"
            if xy_path.exists():
                xy_path.unlink()

            # Seafloor age
            link_path = raster_output_dir / "SeafloorAge" / f"seafloor_age_{time:0.0f}.nc"
            if link_path.exists() and link_path.is_symlink():
                link_path.unlink()

            # Bathymetry
            link_path = carbonate_workdir / f"paleobathymetry_{time:0.0f}.nc"
            if link_path.exists() and link_path.is_symlink():
                link_path.unlink()

## Crustal thickness

In [ ]:
if use_features('crustal'):
    crust_output_dir = raster_output_dir / "CrustalThickness"
    crust_output_dir.mkdir(parents=True, exist_ok=True)

    run_crust = overwrite
    if not run_crust:
        for time in times:
            fp = crust_output_dir / f"crustal_thickness_{time:0.0f}Ma.nc"
            if not fp.exists():
                run_crust = True
                break

    crust_workdir = raster_output_dir / "crust_outputs"
    if run_crust:
        crust_workdir.mkdir(parents=True, exist_ok=True)

### Paleotopography

In [ ]:
if use_features('crustal'):
    # Setup
    area_threshold = 0.0
    subdivision_depth = 2
    resolution = 0.5  # degrees
    ocean_elevation = -1000.0
    shallow_elevation = -200.0
    land_elevation = 200.0
    mountain_relief = 3100.0
    mountain_buffer_distance = 2.0  # degrees

    run_paleotopo = overwrite
    if not run_paleotopo:
        for time in times:
            fp = crust_workdir / f"paleotopo_{resolution:0.2f}d_{time:0.2f}Ma.nc"
            if not fp.exists():
                run_paleotopo = True
                break

In [ ]:
if use_features('crustal'):
    if run_paleotopo:
        paleotopo_dir = crust_workdir / "paleotopography-data"
        cookie_cut_dir = crust_workdir / "cookie_cut"
        # Download and extract paleotopography data
        from lib.check_files import check_paleotopography_data
        check_paleotopography_data(
            data_dir=paleotopo_dir,
            verbose=True,
        )

        # Load polygons
        polygons_dir = paleotopo_dir / "Paleogeography_Matthews2016_410-2Ma_Shapefiles"
        
        gdf_paleotopo = []
        for d in polygons_dir.iterdir():
            if not d.is_dir():
                continue
            for fname in d.iterdir():
                if not fname.name.endswith(".shp"):
                    continue
                which = fname.name.split("_")[0]
                gdf = gpd.read_file(fname)
                gdf["NAME"] = which
                gdf["ENV"] = which
                gdf_paleotopo.append(gdf)
        gdf_paleotopo = pd.concat(gdf_paleotopo, ignore_index=True).dropna(axis="columns")
        # gdf_paleotopo = gdf_paleotopo.drop(columns="PLATEID1").explode()

        # Cookie-cut polygons for plate model
        cookie_cut_dir.mkdir(exist_ok=True)
        sp_file = crust_workdir / "static_polygons.shp"
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            pygplates.reconstruct(
                plate_model.static_polygons,
                plate_model.rotation_model,
                str(sp_file),
                0.0,
            )
        gdf_sp = gpd.read_file(sp_file).explode()
        gdf_sp = gdf_sp[gdf_sp.geometry.type == "Polygon"]
        gdf_sp = gdf_sp[["PLATEID1", gdf_sp.geometry.name]]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            gdf_paleotopo = gpd.overlay(
                gdf_paleotopo,
                gdf_sp,
                how="intersection",
            )
        inds = np.squeeze(np.where(
            (gdf_paleotopo["PLATEID1_2"].str.startswith("5"))
            | (gdf_paleotopo["PLATEID1_1"] == 616)
        ))
        for ind in inds:
            gdf_paleotopo.at[ind, "PLATEID1_2"] = gdf_paleotopo.at[ind, "PLATEID1_1"]
        gdf_paleotopo = gdf_paleotopo.rename(
            columns={"PLATEID1_2": "PLATEID1"}
        ).drop(columns="PLATEID1_1")

        # gdf_paleotopo = gpd.overlay(
        #     gdf_paleotopo,
        #     gdf_sp,
        #     how="intersection",
        #     keep_geom_type=True,
        # )
        for which in ("i", "sm", "m", "lm"):
            shp_path = cookie_cut_dir / f"{which}_402_2.shp"
            gdf_which = gdf_paleotopo[gdf_paleotopo["ENV"] == which]
            gdf_which.to_file(str(shp_path))
    else:
        gdf_paleotopo = []
        for which in ("i", "sm", "m", "lm"):
            shp_path = cookie_cut_dir / f"{which}_402_2.shp"
            gdf_paleotopo.append(gpd.read_file(shp_path))
        gdf_paleotopo = pd.concat(gdf_paleotopo, ignore_index=True)

In [ ]:
if use_features('crustal'):
    tween_dir = crust_workdir / "tween_dir"
    tween_dir.mkdir(exist_ok=True)
    topography_filename = str(paleotopo_dir / "topo15_3600x1800.nc")
    classes_filename = str(crust_workdir / "present_day_topo_as_classes.nc")
    presentday_filename = str(crust_workdir / "present_day_paleogeography.gmt")

    if run_paleotopo:
        create_present_day_features(
            topography_filename=topography_filename,
            classes_filename=classes_filename,
            features_filename=presentday_filename,
        )

In [ ]:
if use_features('crustal'):
    # Takes an hour or two, so only run if necessary
    tween_times = sorted(gdf_paleotopo["TIME"].unique())

    run_tween = overwrite
    if not run_tween:
        for t1, t2 in zip(tween_times[:-1], tween_times[1:]):
            for which in (
                "tweentest_land",
                "tweentest_ocean",
                "mountain_regression",
                "mountain_stable",
                "mountain_transgression",
            ):
                fp = tween_dir / f"{which}_{t1:0.2f}Ma_{t2:0.2f}Ma.gpmlz"
                if not fp.exists():
                    run_tween = True
                    break

    if run_tween:
        tween_paleoshorelines_inplace(
            tween_dir=str(tween_dir),
            basedir=str(cookie_cut_dir),
            rotation_filenames=plate_model.rotation_model.filenames,
            presentday_filename=presentday_filename,
            times=tween_times,
            resolution=resolution,
            nprocs=n_jobs,
            verbose=False,
        )

In [ ]:
if use_features('crustal'):
    # Create paleotopography grids
    if run_paleotopo:
        paleogeography_timeslice_list = sorted(gdf_paleotopo["TIME"].unique())
        paleogeography_timeslice_list.append(0.0)
        paleogeography_timeslice_list = np.array(paleogeography_timeslice_list)
        paleogeography_timeslice_list.sort()

        with joblib.Parallel(n_jobs) as parallel:
            parallel(
                joblib.delayed(paleotopography_job)(
                    reconstruction_time=time,
                    paleogeography_timeslice_list=paleogeography_timeslice_list,
                    tween_basedir=tween_dir,
                    reconstruction_basedir=str(crust_workdir / "cookie_cut"),
                    output_dir=crust_workdir,
                    file_format="gpmlz",
                    rotation_file=plate_model.rotation_model.filenames,
                    COBterrane_file=gplot._continents.filenames,
                    agegrid_file_template="",
                    lowland_elevation=land_elevation,
                    shallow_marine_elevation=shallow_elevation,
                    max_mountain_elevation=mountain_relief,
                    depth_for_unknown_ocean=ocean_elevation,
                    sampling=resolution,
                    mountain_buffer_distance_degrees=mountain_buffer_distance,
                    area_threshold=area_threshold,
                    grid_smoothing_wavelength_kms=None,
                    merge_with_bathymetry=False,
                    land_or_ocean_precedence='land',
                    netcdf3_output=False,
                    subdivision_depth=subdivision_depth,
                    present_day_filename=presentday_filename,
                )
                for time in times
            )

### Crustal thickness

In [ ]:
if use_features('crustal'):
    if run_crust:
        with joblib.Parallel(n_jobs) as parallel:
            parallel(
                joblib.delayed(calculate_crustal_thickness)(
                    time=time,
                    input_dir=str(crust_workdir),
                    output_dir=str(crust_output_dir),
                )
                for time in times
            )

## Oceanic crustal CO₂

In [ ]:
if use_features('subduction'):
    co2_output_dir = raster_output_dir / "CrustalCO2"
    co2_output_dir.mkdir(parents=True, exist_ok=True)

    run_co2 = overwrite
    if not run_co2:
        for time in times:
            fp = co2_output_dir / f"crustal_co2_{time:0.0f}Ma.nc"
            if not fp.exists():
                run_co2 = True
                break

    if run_co2:
        calculate_crustal_co2(
            times=times,
            seafloor_age_dir=seafloor_age_output_dir,
            output_dir=co2_output_dir,
            n_jobs=n_jobs,
        )

## Erosion

In [ ]:
if use_features('erodep'):
    erodep_output_dir = raster_output_dir / "ErosionDeposition"
    erodep_output_dir.mkdir(exist_ok=True)

    run_erodep = False
    erodep_times = range(0, 270, 5)
    for time in erodep_times:
        if time <= min_time - 5 or time >= max_time + 5:
            continue
        fp = erodep_output_dir / f"erosion_deposition_{time:0.0f}Ma.nc"
        if not fp.exists():
            run_erodep = True
            break

In [ ]:
if use_features('erodep'):
    if run_erodep:
        erodep_workdir = raster_output_dir / "erodep_outputs"
        erodep_workdir.mkdir(exist_ok=True)

        from lib.check_files import check_erodep_data
        check_erodep_data(
            data_dir=erodep_workdir,
            verbose=True,
            force=True,
        )

        # Copy missing files
        srcdir = erodep_workdir / "erosion_deposition_files"
        for time in erodep_times:
            if time <= min_time - 5 or time >= max_time + 5:
                continue
            src = srcdir / f"erodep{time:0.0f}Ma.nc"
            dst = erodep_output_dir / f"erosion_deposition_{time:0.0f}Ma.nc"
            if dst.exists():
                continue
            shutil.copy2(src, dst)

## Cleanup

Remove all working directories to save space.

In [ ]:
if cleanup:
    for d in (
        "carbonate_outputs",
        "crust_outputs",
        "erodep_outputs",
        "sedthick_outputs",
    ):
        cleanup_dir = raster_output_dir / d
        if cleanup_dir.exists():
            shutil.rmtree(cleanup_dir)